# Instalación y verificación en Google Colab

Este notebook está preparado para ejecutarse en **Google Colab**. El runtime es temporal: bibliotecas, archivos y caché se pierden al reiniciar o desconectar. Por eso no se crea `.venv`; Colab ya proporciona un intérprete aislado.

> Un modelo grande se descarga únicamente cuando una aplicación lo carga por primera vez. Este notebook no inicia esa descarga.

## 1. Configurar el acelerador

En Colab abre **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **GPU** si el proyecto usa PyTorch. Reinicia el runtime si Colab lo solicita. La GPU asignada depende de su disponibilidad.

In [ ]:
import platform
import sys
from pathlib import Path

try:
    from google.colab import files  # noqa: F401
    en_colab = True
except ModuleNotFoundError:
    en_colab = False

print(f'Google Colab: {en_colab}')
print(f'Sistema operativo: {platform.system()} {platform.release()}')
print(f'Python: {sys.version.split()[0]}')
print(f'Directorio de trabajo: {Path.cwd()}')
if sys.version_info < (3, 10):
    raise RuntimeError('Se requiere Python 3.10 o superior.')
print('Entorno compatible')

## 2. Cargar el proyecto

El proyecto se descarga por código desde el ZIP público de la rama `main`; no usa el selector de archivos del navegador. La celda obtiene el ZIP, lo extrae en `/content` y detecta automáticamente `requirements.txt`.

No incluyas claves, tokens ni archivos `.env` al crear o compartir el ZIP.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from urllib.request import urlretrieve

# ZIP público de la rama main. Se descarga por código; no usa el selector del navegador.
URL_ZIP = 'https://github.com/CaMunozS/topicos2/archive/refs/heads/main.zip'
CONTENIDO = Path('/content')
ARCHIVO_ZIP = CONTENIDO / 'topicos2-main.zip'

print(f'Descargando: {URL_ZIP}')
urlretrieve(URL_ZIP, ARCHIVO_ZIP)

with ZipFile(ARCHIVO_ZIP) as zf:
    raices = {Path(nombre).parts[0] for nombre in zf.namelist() if nombre and not nombre.startswith('__MACOSX/')}
    if len(raices) != 1:
        raise RuntimeError(f'El ZIP debe tener una sola carpeta raíz; se encontraron: {raices}')
    zf.extractall(CONTENIDO)

DIRECTORIO_PROYECTO = CONTENIDO / raices.pop()
ruta_requirements = DIRECTORIO_PROYECTO / 'requirements.txt'
if not ruta_requirements.exists():
    raise FileNotFoundError(f'No se encontró requirements.txt en {DIRECTORIO_PROYECTO}.')

RUTA_REQUIREMENTS = str(ruta_requirements)
%cd {DIRECTORIO_PROYECTO}
print(f'Proyecto listo: {DIRECTORIO_PROYECTO}')
print(f'Requirements: {RUTA_REQUIREMENTS}')

## 3. Instalar dependencias

Después de cargar o clonar el proyecto, ejecuta la siguiente celda. Colab normalmente ya incluye PyTorch compatible con su runtime; evita reinstalarlo salvo que el proyecto fije una versión concreta.

In [ ]:
# Ejecuta primero la celda anterior: ella define RUTA_REQUIREMENTS.
if 'RUTA_REQUIREMENTS' not in globals():
    raise RuntimeError('Primero carga y extrae el ZIP del proyecto en la sección 2.')

!pip install -r {RUTA_REQUIREMENTS}

## 4. Token opcional de Hugging Face

El modelo predeterminado es público; `HF_TOKEN` no es necesario para la simulación ni para ese modelo. Solo crea el secreto `HF_TOKEN` en **Secrets** de Colab si eliges un modelo restringido cuya licencia ya aceptaste. El token nunca se escribe en el notebook.

In [ ]:
import os

NOMBRE_TOKEN_HF = 'HF_TOKEN'
token_hf = os.environ.get(NOMBRE_TOKEN_HF)

if en_colab and not token_hf:
    try:
        from google.colab import userdata
        token_hf = userdata.get(NOMBRE_TOKEN_HF)
    except Exception:
        token_hf = None

if token_hf:
    os.environ[NOMBRE_TOKEN_HF] = token_hf
    print('Token de Hugging Face disponible para esta sesión.')
else:
    print('No se configuró HF_TOKEN. Puedes continuar con simulación y con el modelo público predeterminado.')

## 5. Verificar PyTorch y GPU

Si aparece `cpu` aunque elegiste GPU, reinicia el runtime desde el menú de Colab y ejecuta nuevamente el notebook desde el inicio.

In [ ]:
try:
    import torch
    tiene_cuda = torch.cuda.is_available()
    dispositivo = 'cuda' if tiene_cuda else 'cpu'
    print(f'PyTorch: {torch.__version__}')
    print(f'Dispositivo seleccionado: {dispositivo}')
    if tiene_cuda:
        print(f'GPU asignada: {torch.cuda.get_device_name(0)}')
        print(f'CUDA: {torch.version.cuda}')
    else:
        print('No hay GPU disponible. Configura GPU en el menú de Entorno de ejecución si la necesitas.')
except ModuleNotFoundError:
    print('PyTorch no está instalado. Revisa requirements.txt o instala la versión requerida por el proyecto.')

## 6. Ejecutar el proyecto

Ejecuta primero la simulación. Esta celda invoca realmente `run.py`, pero no usa OpenAI, GPU, token de Hugging Face ni descarga el modelo de 4B. Si termina correctamente, el proyecto y sus dependencias están operativos en Colab.

In [ ]:
import subprocess

if 'DIRECTORIO_PROYECTO' not in globals():
    raise RuntimeError('Primero descarga el ZIP del proyecto en la sección 2.')

def ejecutar_con_detalle(argumentos):
    """Ejecuta run.py y muestra cada línea de salida en tiempo real."""
    comando = [sys.executable, '-u', 'run.py', *argumentos]
    print('Comando:', ' '.join(comando))
    proceso = subprocess.Popen(
        comando, cwd=DIRECTORIO_PROYECTO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    )
    for linea in iter(proceso.stdout.readline, ''):
        print(linea, end='')
    codigo = proceso.wait()
    if codigo != 0:
        raise RuntimeError(f'run.py terminó con código de salida {codigo}.')
    print('run.py terminó correctamente.')

ejecutar_con_detalle(['--simulate'])

## 7. Entrenamiento real con OpenAI

La corrida real necesita el secreto `OPENAI_API_KEY` de Colab y puede generar cargos. Está bloqueada por defecto: después de guardar el secreto en **Secrets** y revisar el presupuesto, cambia `EJECUTAR_ENTRENAMIENTO_REAL` a `True`. El ejemplo usa una ronda y un máximo de US$2.

In [ ]:
EJECUTAR_ENTRENAMIENTO_REAL = False

if EJECUTAR_ENTRENAMIENTO_REAL:
    clave_openai = os.environ.get('OPENAI_API_KEY')
    if en_colab and not clave_openai:
        try:
            from google.colab import userdata
            clave_openai = userdata.get('OPENAI_API_KEY')
        except Exception:
            clave_openai = None
    if not clave_openai:
        raise RuntimeError('Falta el secreto OPENAI_API_KEY en Colab.')

    os.environ['OPENAI_API_KEY'] = clave_openai
    if 'ejecutar_con_detalle' not in globals():
        raise RuntimeError('Ejecuta primero la sección 6 para cargar el monitor de salida.')
    ejecutar_con_detalle(['--rounds', '1', '--budget', '2'])
else:
    print('Entrenamiento real no iniciado. Cambia EJECUTAR_ENTRENAMIENTO_REAL a True para invocarlo.')

## 8. Persistencia opcional con Google Drive

Monta Drive únicamente si necesitas conservar resultados, checkpoints o descargas entre sesiones. Evita almacenar secretos dentro del notebook o de un repositorio compartido.

In [ ]:
# Opcional: ejecuta solo en Colab y solo si necesitas persistencia.
if en_colab:
    from google.colab import drive
    # drive.mount('/content/drive')
    print("Descomenta drive.mount('/content/drive') para autorizar y montar tu Drive.")
else:
    print('Google Drive se monta únicamente desde Colab.')

## Checklist final

- [ ] Se seleccionó GPU en Colab, cuando el proyecto la requiere.
- [ ] Se cargó o clonó el proyecto sin secretos.
- [ ] Se instaló `requirements.txt`, si existe.
- [ ] PyTorch detecta `cuda` cuando se requiere aceleración NVIDIA.
- [ ] Los resultados importantes se guardan fuera del runtime temporal.